# GigaAM-v4 + Multilingual-v2 — English, Russian, and 21-language ASR on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/GigaAM-v4-multilingual-asr.ipynb)

Sber's [`ai-sage/GigaAM-v4`](https://huggingface.co/ai-sage/GigaAM-v4) is a **238M** Zapformer speech recognizer for **English and Russian**. [`ai-sage/GigaAM-Multilingual-v2`](https://huggingface.co/ai-sage/GigaAM-Multilingual-v2) is **776M** and covers **21 languages**. Both are [MIT](https://huggingface.co/ai-sage/GigaAM-v4/blob/main/LICENSE), public (not gated), and load with Hugging Face Transformers **custom code** (`trust_remote_code=True`).

**Why these models.** v4 was pretrained on **7 million hours** of unlabelled audio and trained on **58,000 hours** of supervised speech. Multilingual-v2 was pretrained on **10 million hours** and trained on **155,000 hours**. Both revisions used here are dual-head RNN-T: unformatted `raw` text and formatted `e2e` text (punctuation / capitalization). Streaming (160 / 320 / 640 ms) is on the cards; this notebook stays on the cards' full-context `transcribe` call.

On the v4 card, RNN-T Russian average WER is **5.99** and English Open ASR public-subset average WER is **7.28**. On the Multilingual-v2 card, full-context `auto` average WER across 21 languages is **3.96**. Those are the cards' numbers, not a T4 measurement from this notebook.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/GigaAM-v4-multilingual-asr.md) for when this notebook is a good fit and when it is not.

This notebook uses the **🤗 Transformers** path both cards document:

```python
from transformers import AutoModel

model = AutoModel.from_pretrained(
    "ai-sage/GigaAM-v4", revision="rnnt", trust_remote_code=True,
    device="cuda", fp16_encoder=True,
).eval()
result = model.transcribe("example.wav")
print(result["raw"].text)
print(result["e2e"].text)
```

Multilingual-v2 is the same call with `language="auto"` on load and an optional language tag on `transcribe` (`language="fr"` here). `config.json` on both repos records `transformers_version: 4.57.1`. The cards install **`transformers<5`**.

**Licence:** MIT.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~16 GB of VRAM and **no native bfloat16**. The cards' load already uses **`fp16_encoder=True`** (fp16, not bf16). Saved weights are float32: v4 `model.safetensors` is **486.3 MB**; Multilingual-v2 is **1.6 GB**. The notebook loads them **one at a time** and frees VRAM between models. `transcribe` rejects files longer than **30 seconds** (use `transcribe_longform` on the card for those).

**Not in this notebook:** CTC / SSL revisions, streaming (`create_stream`), ONNX export, fine-tuning, or speaker labels.

**Links:** [GigaAM-v4 card](https://huggingface.co/ai-sage/GigaAM-v4) · [Multilingual-v2 card](https://huggingface.co/ai-sage/GigaAM-Multilingual-v2) · [announcement](https://x.com/aitrackerbot/status/2126228109972156712) · [GigaAM GitHub](https://github.com/salute-developers/GigaAM) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/GigaAM-v4-multilingual-asr.md)


## 0. Check the GPU

Confirm CUDA is available, print the device name and free VRAM. Need a T4 (or any CUDA GPU). A restart after the install cell drops this state; section 2 prints it again before load.


In [ ]:
import os
import shutil

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; the cards' fp16_encoder=True is the T4 path.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")
    print("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Install (keep Colab's torch)

Both cards install:

```bash
pip install numpy "transformers<5" onnx onnxruntime hydra-core omegaconf sentencepiece tqdm
```

`config.json` was saved with `transformers_version: 4.57.1`. This notebook pins **`transformers>=4.57.1,<5`**. Custom `modeling_gigaam.py` also imports **`soundfile`** and **`torchaudio`** at load time (torchaudio is already on Colab). `huggingface_hub>=0.34,<1` matches Transformers 4.57.x — do not upgrade hub to 1.x/2.x.

**Do not reinstall torch** — Colab already ships a CUDA wheel; forcing a reinstall often drops GPU support. Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled torch.

**Do not import the model in this cell.** Colab has already imported its preinstalled transformers 5.x. After reconnect, skip this cell and continue from section 2.


In [ ]:
import importlib.metadata as md
import os
import signal
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

INSTALL_TAG = "tf>=4.57.1,<5-gigaam-v4"
MARKER = (
    Path("/content/.gigaam_v4_nb_install_ok")
    if Path("/content").is_dir()
    else Path(".gigaam_v4_nb_install_ok")
)
already_restarted = MARKER.exists() and MARKER.read_text().strip() == INSTALL_TAG

print("python", sys.version.split()[0])
try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")

# Public weights: skip Colab userdata.get("HF_TOKEN") popups.
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

# Card: numpy "transformers<5" onnx onnxruntime hydra-core omegaconf sentencepiece tqdm.
# Keep Colab's CUDA torch. modeling_gigaam.py imports soundfile at module load.
# transformers 4.57.x needs huggingface_hub>=0.34,<1.
get_ipython().run_line_magic(
    "pip",
    'install -q "transformers>=4.57.1,<5" "huggingface_hub>=0.34.0,<1" '
    "numpy onnx onnxruntime hydra-core omegaconf sentencepiece tqdm soundfile",
)

print()
print("transformers", md.version("transformers"))
print("huggingface_hub", md.version("huggingface_hub"))
print("omegaconf", md.version("omegaconf"))
print("sentencepiece", md.version("sentencepiece"))
print("soundfile", md.version("soundfile"))
print("onnx", md.version("onnx"))
print("onnxruntime", md.version("onnxruntime"))
print("hydra-core", md.version("hydra-core"))

MARKER.write_text(INSTALL_TAG)
if already_restarted:
    print("Install marker already set; not restarting again. Continue from section 2.")
else:
    print()
    print("=" * 72)
    print("Restarting the session so transformers>=4.57.1,<5 is imported.")
    print("After reconnect: skip this cell and run from section 2 (runtime check).")
    print("A 'session crashed' or 'Operation cancelled' message here is the restart,")
    print("not a failed install — pip already finished (versions printed above).")
    print("=" * 72)
    os.kill(os.getpid(), signal.SIGKILL)


> The install cell **restarts the session for you**. After reconnect, skip it and continue from section 2.
> A "session crashed" message after install is the restart, not a failed pip. If you restarted by hand instead: *Runtime > Restart session*, then this next cell.


## 2. Runtime check after restart

Confirm `transformers` is 4.57.x (`<5`), CUDA is on, and print the device name and free VRAM. T4 (`sm_75`) has no native bfloat16 — the cards' **`fp16_encoder=True`** is the T4 path.


In [ ]:
import os
import time
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

import torch
import transformers
from packaging.version import Version

ver = Version(transformers.__version__.split("+")[0])
assert ver >= Version("4.57.1") and ver < Version("5"), (
    f"transformers {transformers.__version__} is loaded; GigaAM cards pin transformers<5 "
    "(config.json transformers_version 4.57.1). Run the install cell, then "
    "Runtime > Restart session."
)

print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

major, minor = torch.cuda.get_device_capability()
DEVICE = "cuda"
V4_ID = "ai-sage/GigaAM-v4"
MULTI_ID = "ai-sage/GigaAM-Multilingual-v2"
SR = 16_000

free_b, total_b = torch.cuda.mem_get_info()
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print("load kwargs: device='cuda', fp16_encoder=True  (cards; T4 has no native bf16)")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
if major < 8:
    print("No native bfloat16 on this GPU — staying on the cards' fp16 encoder.")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB | "
        f"free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


def report(result, seconds, label):
    print(f"[{label}] raw: {result['raw'].text}")
    print(f"[{label}] e2e: {result['e2e'].text}")


## 3. Public sample clips

All three files are public and need **no Hugging Face token**.

- **English:** `librispeech_mr_quilter.wav` from [`hf-internal-testing/dummy-audio-samples`](https://huggingface.co/datasets/hf-internal-testing/dummy-audio-samples) (LibriSpeech-style read speech).
- **Russian:** one [`PolyAI/minds14`](https://huggingface.co/datasets/PolyAI/minds14) `ru-RU` phone-banking clip (CC BY 4.0), via the Hugging Face dataset viewer.
- **French:** the same dataset's `fr-FR` split, for Multilingual-v2.

`model.transcribe` loads the wav with ffmpeg and resamples to 16 kHz. Clips here are well under the **30 s** full-context limit on `transcribe`.


In [ ]:
import urllib.request
from pathlib import Path

import requests
from IPython.display import Audio, display

SAMPLE_DIR = Path("gigaam_samples")
SAMPLE_DIR.mkdir(exist_ok=True)
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}

DUMMY = (
    "https://huggingface.co/datasets/hf-internal-testing/dummy-audio-samples/resolve/main"
)
EN_URLS = [
    (f"{DUMMY}/librispeech_mr_quilter.wav", "librispeech_mr_quilter.wav"),
    (f"{DUMMY}/fleurs_en_us_test_0346.wav", "fleurs_en_us_test_0346.wav"),
    ("https://cdn-media.huggingface.co/speech_samples/sample1.flac", "hf-speech-sample1.flac"),
]


def _urlretrieve(url, path, timeout=120):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as resp, open(path, "wb") as f:
        f.write(resp.read())
    return Path(path)


def download_first(urls_and_names):
    last_err = None
    for url, name in urls_and_names:
        dest = SAMPLE_DIR / name
        try:
            if not dest.exists() or dest.stat().st_size == 0:
                print("Downloading", url)
                _urlretrieve(url, dest)
            print(f"loaded {dest.name}: {dest.stat().st_size} bytes")
            return dest
        except Exception as exc:
            last_err = exc
            print(f"failed {url}: {exc!r}")
    raise RuntimeError(f"All English downloads failed ({last_err!r})")


def minds14_clip(config, indexes=(1, 0, 2)):
    last_err = None
    for index in indexes:
        try:
            rows = requests.get(
                "https://datasets-server.huggingface.co/rows",
                params={
                    "dataset": "PolyAI/minds14",
                    "config": config,
                    "split": "train",
                    "offset": index,
                    "length": 1,
                },
                headers=UA,
                timeout=120,
            )
            rows.raise_for_status()
            row = rows.json()["rows"][0]["row"]
            src = row["audio"][0]["src"] if isinstance(row["audio"], list) else row["audio"]["src"]
            dest = SAMPLE_DIR / f"minds14-{config}-{index}.wav"
            if not dest.exists() or dest.stat().st_size == 0:
                print("Downloading", src)
                _urlretrieve(src, dest)
            return dest, row.get("transcription")
        except Exception as exc:
            last_err = exc
            print(f"minds14 {config} offset={index} failed: {exc!r}")
    raise RuntimeError(f"minds14 {config} failed ({last_err!r})")


en_path = download_first(EN_URLS)
ru_path, ru_ref = minds14_clip("ru-RU")
MULTI_LANG, MULTI_LABEL = "fr", "French"
try:
    fr_path, fr_ref = minds14_clip("fr-FR")
except Exception as exc:
    print(f"French minds14 failed ({exc!r}); using public Spanish FLEURS clip")
    fr_path = download_first([(f"{DUMMY}/fleur_es_sample.wav", "fleur_es_sample.wav")])
    fr_ref = None
    MULTI_LANG, MULTI_LABEL = "es", "Spanish"

print("multilingual demo language tag:", MULTI_LANG)
clips = {
    "en": (en_path, None),
    "ru": (ru_path, ru_ref),
    MULTI_LANG: (fr_path, fr_ref),
}
for lang, (path, reference) in clips.items():
    print(lang, path, "| reference:", reference)
    display(Audio(str(path)))


## 4. Load GigaAM-v4 (RNN-T)

The v4 card's call is `AutoModel.from_pretrained("ai-sage/GigaAM-v4", revision="rnnt", trust_remote_code=True, device="cuda", fp16_encoder=True)`. First run downloads **486.3 MB** (`model.safetensors`) plus a **5.4 MB** VAD file. `main` already points at `rnnt`; `revision="rnnt"` matches the card.


In [ ]:
from transformers import AutoModel

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()
model = AutoModel.from_pretrained(
    V4_ID,
    revision="rnnt",
    trust_remote_code=True,
    device=DEVICE,
    fp16_encoder=True,
).eval()
if torch.cuda.is_available():
    torch.cuda.synchronize()
print(f"loaded {V4_ID} (revision=rnnt) in {time.perf_counter() - t0:.1f}s")
print("class:", type(model).__name__)
print("inner:", type(model.model).__name__)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params / 1e6:.1f}M  (card RNN-T: 238M)")
enc_dtype = next(model.model.encoder.parameters()).dtype
print("encoder dtype:", enc_dtype, "| device:", next(model.parameters()).device)
print_vram("after GigaAM-v4 load")


## 5. Demo 1 — English clip with GigaAM-v4

Card usage: `result = model.transcribe("example.wav")`, then `result["raw"].text` and `result["e2e"].text`. v4 is English+Russian; no language tag on this call.


In [ ]:
import soundfile as sf

en_audio, en_sr = sf.read(str(en_path), always_2d=False)
en_seconds = len(en_audio) / float(en_sr)
print(f"English clip: {en_path.name} ({en_seconds:.2f}s, sr={en_sr})")

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
en_out = model.transcribe(str(en_path))
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
rtf = elapsed / en_seconds if en_seconds else float("nan")

report(en_out, en_seconds, "English / GigaAM-v4")
print(f"audio: {en_seconds:.2f}s  |  wall time: {elapsed:.2f}s  |  RTF {rtf:.3f}x")
print_vram("after English clip")


## 6. Demo 2 — Russian clip with GigaAM-v4

Public [`PolyAI/minds14`](https://huggingface.co/datasets/PolyAI/minds14) `ru-RU` clip (CC BY 4.0). Same `model.transcribe(path)` call as English — v4 has no language argument. The minds14 reference has no punctuation; the card's `e2e` head is the formatted transcript.


In [ ]:
ru_audio, ru_sr = sf.read(str(ru_path), always_2d=False)
ru_seconds = len(ru_audio) / float(ru_sr)
print(f"Russian clip: {ru_path.name} ({ru_seconds:.2f}s, sr={ru_sr})")
print("reference:", ru_ref)

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
ru_out = model.transcribe(str(ru_path))
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
rtf = elapsed / ru_seconds if ru_seconds else float("nan")

print()
report(ru_out, ru_seconds, "Russian / GigaAM-v4")
print(f"audio: {ru_seconds:.2f}s  |  wall time: {elapsed:.2f}s  |  RTF {rtf:.3f}x")
print_vram("after Russian clip")


## 7. Free GigaAM-v4 before loading Multilingual-v2

Multilingual-v2's `model.safetensors` is **1.6 GB**. Delete v4 and empty the CUDA cache so both models are not resident at once on a 16 GB T4.


In [ ]:
import gc

print_vram("before freeing GigaAM-v4")
del model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print_vram("after freeing GigaAM-v4")


## 8. Load GigaAM-Multilingual-v2 (RNN-T)

The Multilingual-v2 card's call is `AutoModel.from_pretrained("ai-sage/GigaAM-Multilingual-v2", revision="rnnt", trust_remote_code=True, language="auto", device="cuda", fp16_encoder=True)`. First run downloads **1.6 GB**. Supported language tags on the card: `ru`, `en`, `kk`, `ky`, `uz`, `be`, `fr`, `es`, `ba`, `ka`, `hy`, `ro`, `tk`, `az`, `tg`, `de`, `it`, `nl`, `pl`, `pt`, `tt` (21 languages) plus `auto`.


In [ ]:
t0 = time.perf_counter()
model = AutoModel.from_pretrained(
    MULTI_ID,
    revision="rnnt",
    trust_remote_code=True,
    language="auto",
    device=DEVICE,
    fp16_encoder=True,
).eval()
if torch.cuda.is_available():
    torch.cuda.synchronize()
print(f"loaded {MULTI_ID} (revision=rnnt, language=auto) in {time.perf_counter() - t0:.1f}s")
print("class:", type(model).__name__)
print("inner:", type(model.model).__name__)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params / 1e6:.1f}M  (card: 776M)")
enc_dtype = next(model.model.encoder.parameters()).dtype
print("encoder dtype:", enc_dtype, "| device:", next(model.parameters()).device)
langs = list(model.config.cfg.get("language_conditioning", {}).get("languages", []))
print(f"{len(langs)} language tags:", ", ".join(langs))
print_vram("after Multilingual-v2 load")


## 9. Demo 3 — non-English clip with Multilingual-v2

Card usage: `result = model.transcribe("example.wav", language="ru")` — pass a supported language tag. This cell uses **`language="fr"`** on a [`PolyAI/minds14`](https://huggingface.co/datasets/PolyAI/minds14) `fr-FR` clip (or `"es"` if that download failed and the public Spanish FLEURS sample was used). Load used `language="auto"`; `transcribe(..., language=...)` overrides it for this file.


In [ ]:
fr_audio, fr_sr = sf.read(str(fr_path), always_2d=False)
fr_seconds = len(fr_audio) / float(fr_sr)
print(f"{MULTI_LABEL} clip: {fr_path.name} ({fr_seconds:.2f}s, sr={fr_sr})")
print("language tag:", MULTI_LANG)
print("reference:", fr_ref)

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
fr_out = model.transcribe(str(fr_path), language=MULTI_LANG)
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
rtf = elapsed / fr_seconds if fr_seconds else float("nan")

print()
report(fr_out, fr_seconds, f"{MULTI_LABEL} / Multilingual-v2")
print(f"audio: {fr_seconds:.2f}s  |  wall time: {elapsed:.2f}s  |  RTF {rtf:.3f}x")
print_vram(f"after {MULTI_LABEL} clip")


## 10. Caveats and cleanup

**What this notebook is for.** The cards' Transformers `AutoModel` + `transcribe` path on a free T4: English and Russian with GigaAM-v4, then a French clip with Multilingual-v2, with wall time printed and VRAM freed between checkpoints.

**What it is not.** Streaming (`create_stream` / 160–640 ms), ONNX, CTC, SSL embeddings, or speaker labels. Pair with [Nemotron-3 Diarization](Nemotron-3-Diarization.ipynb) if you need who-spoke-when. `transcribe` raises on audio longer than 30 seconds — the card's `transcribe_longform` is the VAD-segmented path.

**Languages.** v4: English and Russian. Multilingual-v2 (card): Russian, English, German, Spanish, French, Italian, Dutch, Polish, Portuguese, Romanian, Azerbaijani, Bashkir, Belarusian, Armenian, Georgian, Kyrgyz, Kazakh, Tajik, Turkmen, Tatar, Uzbek.

**T4 notes.** `fp16_encoder=True` as on the cards (fp16, not bf16). Load one model at a time.

**Related notebooks:** [Rumble-2-Flash](Rumble-2-Flash-noisy-speech-to-text.ipynb), [Fluid Speech Pro](Fluid-Speech-Pro-multilingual-asr.ipynb), [Nemotron-Labs Unified ASR 0.6B](Nemotron-Labs-Unified-ASR-0.6B-multilingual.ipynb), [Granite Speech 5.0 2B](granite-speech-5.0-2b-asr-translation.ipynb), [oruk-loop-153M](oruk-loop-153M-looped-asr.ipynb), [Whisper](AIBackends-Whisper-audio-transcription.ipynb).

Free the GPU before you open another notebook:


In [ ]:
import gc

print_vram("before cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
del model
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
